# MCP server 기능을 AgentCore Gateway와 동기화하기

## 개요

MCP server의 도구, 프롬프트 및 리소스 정의는 시간이 지나면서 변경됩니다. AgentCore Gateway는 각 MCP server 대상이 실제로 제공하는 항목과 카탈로그를 동기화하기 위한 세 가지 메커니즘을 제공합니다.

1. **명시적 동기화** — 업스트림 MCP server가 변경된 후 필요할 때 `SynchronizeGatewayTargets`를 호출합니다.
2. **암시적 동기화** — `CreateGatewayTarget`와 `UpdateGatewayTarget`는 작업의 일부로 항상 업스트림 서버의 카탈로그를 다시 읽습니다.
3. **동적 목록 조회** (`listingMode='DYNAMIC'`) — Gateway가 모든 목록 요청(`tools/list`, `prompts/list`, `resources/list`, `resources/templates/list`)을 MCP server로 전달하므로 동기화가 필요하지 않습니다.

> **(1)과 (2)가 (3)과 맺는 관계.** 명시적 동기화와 암시적 동기화는 모두 **`listingMode='DEFAULT'` 대상에 대한 컨트롤 플레인 작업**입니다. 즉, DEFAULT 모드의 목록 호출에 응답할 때 사용할 AgentCore Gateway *캐시*를 채웁니다. `CreateGatewayTarget`는 최초로 캐시를 채우고(생성 시 암시적으로 수행), `UpdateGatewayTarget`는 업데이트할 때마다 부수 효과로 캐시를 다시 채우며, `SynchronizeGatewayTargets`는 그 사이에 필요할 때 캐시를 다시 채웁니다. DYNAMIC 모드 대상은 생성 또는 업데이트 작업 중에 캐시되지 않습니다. 

## 워크숍 진행 순서

| 단계 | 수행할 작업 |
|---|---|
| **1** | Notebook 환경(환경 변수, 유틸리티, 로깅)을 설정합니다. |
| **2** | Cognito 인바운드 인증, IAM 역할, Gateway 순서로 AgentCore Gateway를 생성합니다. |
| **3** | 초기 FastMCP server(현재는 `getOrder`와 `updateOrder`만 포함)를 AgentCore Runtime에 배포합니다. |
| **4** | MCP Server를 Gateway 대상으로 연결합니다(아웃바운드 OAuth, 대상 생성, 인바운드 토큰, `GatewayMCPClient` 헬퍼). |
| **5** | **명시적 동기화**를 실습합니다. 도구를 추가하고 다시 배포한 후, `SynchronizeGatewayTargets`를 실행할 때까지 Gateway 카탈로그가 이전 상태로 유지되는지 확인합니다. |
| **6** | **암시적 동기화**를 실습합니다. 도구를 하나 더 추가하고 다시 배포한 다음 `UpdateGatewayTarget`를 호출하여 부수 효과로 카탈로그가 새로 고쳐지는지 확인합니다. |
| **7** | **동적 목록 조회**를 실습합니다. `listingMode='DYNAMIC'`을 사용하는 두 번째 대상을 생성하고 도구 목록 조회 작업에서 캐시된 결과와 실시간 결과를 비교합니다. |
| **8** | 리소스를 정리합니다. |

## 튜토리얼 세부 정보

| 정보                 | 세부 정보                                                            |
|:---------------------|:---------------------------------------------------------------------|
| 튜토리얼 유형        | 대화형                                                               |
| AgentCore 구성 요소  | AgentCore Gateway, AgentCore Identity, AgentCore Runtime             |
| 에이전틱 프레임워크  | Strands Agents                                                       |
| Gateway 대상 유형    | MCP server                                                           |
| MCP 기본 요소        | 도구, 프롬프트, 리소스(정적 및 템플릿 기반)                          |
| 인바운드 인증 IdP    | Amazon Cognito(다른 서비스도 사용 가능)                              |
| 아웃바운드 인증      | Amazon Cognito(다른 서비스도 사용 가능)                              |
| LLM 모델             | Anthropic Claude Haiku 4.5                                           |
| 튜토리얼 구성 요소   | 명시적 동기화, 암시적 동기화, 동적 목록 조회                        |
| 튜토리얼 분야        | 분야 공통                                                            |
| 예제 난이도          | 쉬움                                                                 |
| 사용 SDK             | boto3                                                                |

### 1단계: 설정 및 사전 요구 사항

이 튜토리얼을 실행하려면 다음 항목이 필요합니다.
* Jupyter notebook(Python 3.10 이상 커널)
* Node.js + npm — AgentCore CLI용(`@aws/agentcore`, 아래 셀에서 전역으로 설치)
* `aws configure`, 환경 변수 또는 인스턴스 역할을 통해 구성한 AWS 자격 증명 및 리전
* CloudFormation, Cognito IDP, IAM 및 Bedrock AgentCore(컨트롤 + 런타임)에 대한 IAM 권한

> Cognito 스택(`agentcore-gateway-lab`)을 `01-mcp-server-target.ipynb`와 공유합니다. `deploy_cognito_stack`은 기존 스택을 멱등적으로 재사용하므로 어느 Notebook을 먼저 실행해도 됩니다. 각 Notebook은 고유한 이름으로 자체 Gateway, IAM 역할 및 MCP server 에이전트를 생성하므로 리소스 충돌이 발생하지 않습니다.

In [ ]:
# 현재 디렉터리의 requirements 파일 또는 pyproject.toml 파일에서 설치
!pip install --force-reinstall -U -r requirements.txt --quiet

In [ ]:
!npm install -g @aws/agentcore

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# 유틸리티 가져오기
import utils
import logging
import boto3
import json
from time import sleep

# Notebook 환경의 로깅 구성
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    handlers=[logging.StreamHandler()],
)

# 특정 로거 수준 설정
logging.getLogger("gateway").setLevel(logging.INFO)

REGION = boto3.Session().region_name
COGNITO_STACK_NAME = "agentcore-gateway-lab"
TEMPLATE_PATH = "cloudformation/cognito-signup-stack.yaml"
MCP_SERVER_NAME = "lab2sync"
GATEWAY_NAME = "ac-gateway-mcp-server-sync"

cfn = boto3.client("cloudformation", region_name=REGION)
cognito = boto3.client("cognito-idp", region_name=REGION)

## 2단계: AgentCore Gateway 생성

### 2.1단계: CloudFormation을 통해 Cognito 배포

[`cloudformation/cognito-signup-stack.yaml`](cloudformation/cognito-signup-stack.yaml)을 배포합니다.

참고: 이 실습에서는 AgentCore Gateway 패턴에 집중하기 위해 인바운드 인증에 Cognito를 사용하도록 AgentCore Gateway를 구성합니다. 엔터프라이즈 워크로드에서는 Entra ID, Auth0, Okta 등 OAuth 2.0 호환 자격 증명 공급자를 인바운드 인증에 사용할 수 있습니다. 자세한 내용은 [자격 증명 공급자 설정](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/identity-idps.html)을 참조하세요. AgentCore Gateway와 대상 간의 아웃바운드 권한 부여에는 [AgentCore Gateway Identity 자격 증명 관리](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/what-is-bedrock-agentcore.html)를 설정하는 것이 좋습니다.

In [ ]:
outputs = utils.deploy_cognito_stack(cfn, COGNITO_STACK_NAME, TEMPLATE_PATH)

# Gateway 인바운드
gw_user_pool_id = outputs["UserPoolId"]
gw_client_id = outputs["GatewayClientId"]
gw_cognito_discovery_url = outputs["DiscoveryUrl"]
scopeString = outputs["GatewayScope"]
token_endpoint = outputs["TokenEndpoint"]
gw_client_secret = cognito.describe_user_pool_client(
    UserPoolId=gw_user_pool_id, ClientId=gw_client_id
)["UserPoolClient"]["ClientSecret"]

# MCP server로의 아웃바운드(동일한 풀)
runtime_user_pool_id = gw_user_pool_id
runtime_client_id = outputs["MCPClientId"]
runtime_cognito_discovery_url = gw_cognito_discovery_url
runtimeScopeString = outputs["MCPScope"]
runtime_client_secret = cognito.describe_user_pool_client(
    UserPoolId=runtime_user_pool_id, ClientId=runtime_client_id
)["UserPoolClient"]["ClientSecret"]

print(f"Stack:              {COGNITO_STACK_NAME}")
print(f"User Pool ID:       {gw_user_pool_id}")
print(f"Discovery URL:      {gw_cognito_discovery_url}")
print(f"Token endpoint:     {token_endpoint}")
print(f"Gateway client ID:  {gw_client_id}")
print(f"MCP client ID:      {runtime_client_id}")
print(f"Gateway scope:      {scopeString}")
print(f"MCP scope:          {runtimeScopeString}")

### 2.2단계: AgentCore Gateway IAM 역할 생성

In [ ]:
agentcore_gateway_iam_role = utils.create_agentcore_gateway_role_with_region(
    GATEWAY_NAME, REGION
)
print("AgentCore Gateway role ARN:", agentcore_gateway_iam_role["Role"]["Arn"])

### 2.3단계: AgentCore Gateway 생성

In [ ]:
gateway_client = boto3.client("bedrock-agentcore-control", region_name=REGION)

auth_config = {
    "customJWTAuthorizer": {
        "allowedClients": [gw_client_id],
        "discoveryUrl": gw_cognito_discovery_url,
    }
}

# 중요: 7단계의 DYNAMIC 대상에는 searchType="NONE"이 필요합니다.
# gateway-target-MCPservers.html에 따르면 시맨틱 검색이 활성화된 Gateway에서는
# DYNAMIC MCP 대상(listingMode=DYNAMIC)이 지원되지 않습니다.
# Notebook 01은 DYNAMIC 대상을 생성하지 않으므로 searchType="SEMANTIC"을
# 사용합니다.
create_response = gateway_client.create_gateway(
    name=GATEWAY_NAME,
    roleArn=agentcore_gateway_iam_role["Role"]["Arn"],
    protocolType="MCP",
    protocolConfiguration={"mcp": {"supportedVersions": ["2025-11-25"]}},
    authorizerType="CUSTOM_JWT",
    authorizerConfiguration=auth_config,
    description="AgentCore Gateway with MCP Server target (sync demos)",
)
gatewayID = create_response["gatewayId"]
gatewayURL = create_response["gatewayUrl"]
print(f"Gateway ID:  {gatewayID}")
print(f"Gateway URL: {gatewayURL}")

### 3단계: AgentCore Runtime에 MCP Server 배포

### 3.1단계: MCP server 코드 확인 및 에이전트 등록

MCP server 코드는 [`mcpservers/app/labsync/main.py`](mcpservers/app/labsync/main.py)에 있습니다. 5단계와 6단계의 동기화 데모에서는 이 파일에 도구를 *추가*하고 Gateway가 변경 사항을 인식하는 시점을 확인합니다. 먼저 두 개의 도구만 사용하여 시작합니다.

In [ ]:
from IPython.display import Code

Code("mcpservers/app/labsync/main.py", language="python")

In [ ]:
!cd mcpservers && agentcore add agent \
    --name {MCP_SERVER_NAME} \
    --type byo \
    --language Python \
    --protocol MCP \
    --code-location app/labsync \
    --authorizer-type CUSTOM_JWT \
    --discovery-url {runtime_cognito_discovery_url} \
    --allowed-clients {runtime_client_id} \
    --allowed-scopes {runtimeScopeString}

### 3.2단계: AgentCore CLI를 통해 배포

In [ ]:
!cd mcpservers && agentcore deploy

In [ ]:
agent = utils.get_agent_status(MCP_SERVER_NAME)

mcp_arn = agent["identifier"]
mcp_url = agent["invocationUrl"]
mcp_id = mcp_arn.split("/")[-1]

print(f"mcp_arn: {mcp_arn}")
print(f"mcp_id:  {mcp_id}")
print(f"mcp_url: {mcp_url}")

## 4단계: MCP Server를 Gateway 대상으로 연결

### 4.1단계: 아웃바운드 인증 구성(OAuth2 자격 증명 공급자)

Gateway가 Cognito에서 발급한 전달자 토큰으로 런타임의 MCP server를 호출하려면 OAuth2 자격 증명 공급자가 필요합니다.

In [ ]:
identity_client = boto3.client("bedrock-agentcore-control", region_name=REGION)

cognito_provider = identity_client.create_oauth2_credential_provider(
    name=f"{GATEWAY_NAME}-identity",
    credentialProviderVendor="CustomOauth2",
    oauth2ProviderConfigInput={
        "customOauth2ProviderConfig": {
            "oauthDiscovery": {"discoveryUrl": runtime_cognito_discovery_url},
            "clientId": runtime_client_id,
            "clientSecret": runtime_client_secret,
        }
    },
)
cognito_provider_arn = cognito_provider["credentialProviderArn"]
print(cognito_provider_arn)

### 4.2단계: Gateway 대상 생성

In [ ]:
create_gateway_target_response = gateway_client.create_gateway_target(
    name="mcp-server-target",
    gatewayIdentifier=gatewayID,
    targetConfiguration={"mcp": {"mcpServer": {"endpoint": mcp_url}}},
    credentialProviderConfigurations=[
        {
            "credentialProviderType": "OAUTH",
            "credentialProvider": {
                "oauthCredentialProvider": {
                    "providerArn": cognito_provider_arn,
                    "scopes": [runtimeScopeString],
                }
            },
        },
    ],
    # AgentCore Runtime이 요청을 특정 microvm에 고정할 수 있도록 클라이언트가
    # 제공한 `Mcp-Session-Id`를 양방향으로 런타임에 전달합니다.
    metadataConfiguration={
        "allowedRequestHeaders": ["Mcp-Session-Id"],
        "allowedResponseHeaders": ["Mcp-Session-Id"],
    },
)
gatewayTargetID = create_gateway_target_response["targetId"]
print(f"Created target: {gatewayTargetID}")

### 4.3단계: Gateway 대상이 READY 상태인지 확인

In [ ]:
list_targets_response = gateway_client.list_gateway_targets(gatewayIdentifier=gatewayID)
print(list_targets_response)

### 4.4단계: 인바운드 액세스 토큰 가져오기

In [ ]:
token_response = utils.get_token(
    token_endpoint, gw_client_id, gw_client_secret, scopeString
)
token = token_response["access_token"]
print("Token (truncated):", token[:60], "...")

### 4.5단계: `GatewayMCPClient` 헬퍼 설정

Notebook과 같은 위치에 정의된 `gateway_mcp_client.GatewayMCPClient`는 전달자 토큰, `MCP-Protocol-Version`, JSON-RPC 연결 처리를 래핑하므로 데모 셀에서 `mcp.list_tools()` 등을 한 줄로 호출할 수 있습니다. 여기에서 한 번 생성한 후 워크숍의 나머지 과정에서 계속 재사용합니다.

In [ ]:
import uuid
from gateway_mcp_client import GatewayMCPClient


def _get_inbound_token() -> str:
    return utils.get_token(token_endpoint, gw_client_id, gw_client_secret, scopeString)[
        "access_token"
    ]


session_id = str(uuid.uuid4())

In [ ]:
mcp = GatewayMCPClient(gatewayURL, _get_inbound_token, session_id=session_id)

print(json.dumps(mcp.list_tools(), indent=2))

## 5단계: `SynchronizeGatewayTargets`를 사용한 명시적 동기화

### 5.1단계: 배경

`SynchronizeGatewayTargets`는 **`listingMode='DEFAULT'` 대상의 카탈로그 캐시를 다시 채우는 컨트롤 플레인 작업**입니다. Gateway는 MCP server와 세션을 열고 카탈로그(도구, 프롬프트, 리소스, 리소스 템플릿)를 가져와 처리하며, 이름 충돌을 방지하기 위해 도구/프롬프트 이름 앞에 대상 이름을 붙이고 영구 인덱스를 업데이트합니다.

이 작업은 *캐시*를 채우므로 DEFAULT 모드 대상에만 의미가 있습니다. DYNAMIC 모드 대상은 캐시를 읽지 않으므로 해당 대상에 `SynchronizeGatewayTargets`를 호출할 필요가 없습니다.

아래에서는 [`mcpservers/app/labsync/main.py`](mcpservers/app/labsync/main.py)을 업데이트하여 새 도구(`cancelOrder`)를 추가하고 다시 배포합니다. 그런 다음 Gateway의 도구 목록에 이 도구가 아직 포함되지 않는지(캐시가 이전 상태인지) 확인하고, `SynchronizeGatewayTargets`를 호출하여 새 도구가 나타나는지 살펴봅니다.

![다이어그램](images/mcp-server-target-explicit-sync.png)

### 5.2단계: MCP server 업데이트(`cancelOrder` 추가)

In [ ]:
%%writefile mcpservers/app/labsync/main.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP(host="0.0.0.0", stateless_http=True)

@mcp.tool()
def getOrder() -> int:
    """Get an order"""
    return 123

@mcp.tool()
def updateOrder(orderId: int) -> int:
    """Update existing order"""
    return 456

@mcp.tool()
def cancelOrder(orderId: int) -> int:
    """cancel existing order"""
    return 789

if __name__ == "__main__":
    mcp.run(transport="streamable-http")

### 5.3단계: 런타임 다시 배포

In [ ]:
print("Re-deploying the MCP server with the live additions...")
!cd mcpservers && agentcore deploy

### 5.4단계: Gateway를 통해 도구 목록 조회 — 여전히 이전 상태

`tools/list`를 호출합니다. 도구를 추가하기 전에 Gateway 카탈로그가 마지막으로 동기화되었으므로 새 `cancelOrder`는 아직 나타나지 않아야 합니다.

In [ ]:
session_id = str(uuid.uuid4())

mcp = GatewayMCPClient(gatewayURL, _get_inbound_token, session_id=session_id)

print(json.dumps(mcp.list_tools(), indent=2))

### 5.5단계: `SynchronizeGatewayTargets` 호출

In [ ]:
sync_response = gateway_client.synchronize_gateway_targets(
    gatewayIdentifier=gatewayID,
    targetIdList=[gatewayTargetID],
)
print(sync_response)

### 5.6단계: 도구 목록 다시 조회 — 동기화로 새 도구 반영

In [ ]:
session_id = str(uuid.uuid4())

mcp = GatewayMCPClient(gatewayURL, _get_inbound_token, session_id=session_id)

sleep(10)
print(json.dumps(mcp.list_tools(), indent=2))

## 6단계: `UpdateGatewayTarget`를 사용한 암시적 동기화

### 6.1단계: 배경

`CreateGatewayTarget`와 `UpdateGatewayTarget`도 **`listingMode='DEFAULT'` 대상에 대한 컨트롤 플레인 작업**이며, `SynchronizeGatewayTargets`와 동일하게 카탈로그를 다시 채웁니다. 단, 생성/업데이트 호출에 이 작업이 함께 포함됩니다. `CreateGatewayTarget`는 새 대상의 캐시를 최초로 채우고, `UpdateGatewayTarget`는 업데이트할 때마다 자동적인 부수 효과로 캐시를 다시 채웁니다. 이후 별도의 동기화 호출은 필요하지 않습니다.

명시적 동기화와 마찬가지로 DEFAULT 모드 대상에만 의미가 있습니다. DYNAMIC 대상에는 채울 캐시가 없습니다.

아래에서는 MCP server에 `deleteOrder`를 추가하고 다시 배포한 다음, 기존 대상에 `UpdateGatewayTarget`를 호출합니다. 카탈로그 새로 고침은 암시적으로 수행됩니다.

![다이어그램](images/mcp-server-target-implicit-sync.png)

### 6.2단계: MCP server 업데이트(`deleteOrder` 추가)

In [ ]:
%%writefile mcpservers/app/labsync/main.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP(host="0.0.0.0", stateless_http=True)

@mcp.tool()
def getOrder() -> int:
    """Get an order"""
    return 123

@mcp.tool()
def updateOrder(orderId: int) -> int:
    """Update existing order"""
    return 456

@mcp.tool()
def cancelOrder(orderId: int) -> int:
    """cancel existing order"""
    return 789

@mcp.tool()
def deleteOrder(orderId: int) -> int:
    """delete existing order"""
    return 101
    
if __name__ == "__main__":
    mcp.run(transport="streamable-http")

### 6.3단계: 런타임 다시 배포

In [ ]:
print("Re-deploying the MCP server with the live additions...")
!cd mcpservers && agentcore deploy

### 6.4단계: `UpdateGatewayTarget` 호출 — 암시적 동기화

In [ ]:
update_gateway_target_response = gateway_client.update_gateway_target(
    gatewayIdentifier=gatewayID,
    targetId=gatewayTargetID,
    name="mcp-server-target",
    targetConfiguration={"mcp": {"mcpServer": {"endpoint": mcp_url}}},
    credentialProviderConfigurations=[
        {
            "credentialProviderType": "OAUTH",
            "credentialProvider": {
                "oauthCredentialProvider": {
                    "providerArn": cognito_provider_arn,
                    "scopes": [runtimeScopeString],
                }
            },
        },
    ],
)
print(update_gateway_target_response)

### 6.5단계: 도구 목록 다시 조회 — 암시적 동기화로 새 도구 반영

In [ ]:
sleep(10)
session_id = str(uuid.uuid4())
mcp = GatewayMCPClient(gatewayURL, _get_inbound_token, session_id=session_id)

print(json.dumps(mcp.list_tools(), indent=2))

## 7단계: `listingMode='DYNAMIC'`을 사용한 동적 목록 조회

### 7.1단계: 배경 — DEFAULT와 DYNAMIC 비교

기본적으로 AgentCore Gateway는 대상이 생성, 업데이트 또는 마지막으로 동기화될 때 검색한 기능(도구, 프롬프트, 리소스, 리소스 템플릿)을 *캐시*합니다. `listingMode='DEFAULT'`에서는 **업스트림 MCP server를 호출하지 않고** Gateway의 카탈로그에서 네 가지 MCP 목록 작업에 응답합니다. 빠르고 복원력이 뛰어나지만 다음 동기화까지 이전 상태가 유지됩니다.

`listingMode='DYNAMIC'`에서는 모든 목록 요청이 업스트림 MCP server로 전달되며 동기화가 필요하지 않습니다.

다음 사항에 유의하세요.

- DYNAMIC 모드는 시맨틱 검색(`x_amz_bedrock_agentcore_search`) 또는 아웃바운드 3자 OAuth(3LO)와 **상호 운용되지 않습니다**.
- DYNAMIC 모드는 도구, 프롬프트, 리소스 및 리소스 템플릿의 네 가지 기본 요소 유형에 모두 동일하게 적용됩니다.

### 7.2단계: 프롬프트, 리소스 및 리소스 템플릿으로 MCP server 확장

**네 가지** 목록 작업 모두에서 DEFAULT와 DYNAMIC의 차이를 보여주려면 업스트림 MCP server가 네 가지 기본 요소 유형을 모두 제공해야 합니다. [`mcpservers/app/labsync/main.py`](mcpservers/app/labsync/main.py)을 다시 작성하여 기존 도구와 함께 프롬프트 및 리소스를 추가하고, 캐시된 결과와 실시간 결과의 차이를 도구 측면에서도 확인할 수 있도록 새 도구 `archiveOrder`를 추가합니다.

In [ ]:
%%writefile mcpservers/app/labsync/main.py
import json

from mcp.server.fastmcp import FastMCP

mcp = FastMCP(host="0.0.0.0", stateless_http=True)


@mcp.tool()
def getOrder() -> int:
    """Get an order"""
    return 123


@mcp.tool()
def updateOrder(orderId: int) -> int:
    """Update existing order"""
    return 456


@mcp.tool()
def cancelOrder(orderId: int) -> int:
    """Cancel existing order"""
    return 789


@mcp.tool()
def deleteOrder(orderId: int) -> int:
    """Delete existing order"""
    return 101


@mcp.tool()
def archiveOrder(orderId: int) -> int:
    """Archive existing order"""
    return 202


if __name__ == "__main__":
    mcp.run(transport="streamable-http")

### 7.3단계: Runtime 다시 배포


In [ ]:
print("Re-deploying the MCP server with the live additions...")
!cd mcpservers && agentcore deploy

### 7.4단계: `listingMode='DYNAMIC'`을 사용하여 새 Gateway 대상 생성

기존 `mcp-server-target`(기본 `listingMode='DEFAULT'` 사용)을 변경하지 않고 *별도의* 대상을 생성하여 두 모드가 동일한 Gateway에 공존하도록 하고 나란히 비교합니다. 두 대상은 동일한 업스트림 MCP server URL을 가리키지만 캐시에서 읽는지, 실시간 서버에서 읽는지에 따라 서로 다른 기능을 보고합니다.

In [ ]:
create_dynamic_target_response = gateway_client.create_gateway_target(
    name="mcp-server-target-dynamic",
    gatewayIdentifier=gatewayID,
    targetConfiguration={
        "mcp": {
            "mcpServer": {
                "endpoint": mcp_url,
                "listingMode": "DYNAMIC",
            }
        }
    },
    credentialProviderConfigurations=[
        {
            "credentialProviderType": "OAUTH",
            "credentialProvider": {
                "oauthCredentialProvider": {
                    "providerArn": cognito_provider_arn,
                    "scopes": [runtimeScopeString],
                }
            },
        },
    ],
    metadataConfiguration={
        "allowedRequestHeaders": ["Mcp-Session-Id"],
        "allowedResponseHeaders": ["Mcp-Session-Id"],
    },
)
dynamicTargetID = create_dynamic_target_response["targetId"]
print(f"Created DYNAMIC target: {dynamicTargetID}")

### 7.5단계: 도구 목록 나란히 조회(실시간 변경 전)

현재 두 대상은 동일한 MCP server URL을 가리킵니다. DEFAULT 대상의 카탈로그는 위의 5단계와 6단계에서 마지막으로 동기화된 상태입니다. DYNAMIC 대상은 방금 생성되었으며 목록을 호출할 때마다 기능을 실시간으로 가져옵니다.

> **페이지 매김은 대상별로 적용됩니다.** 여러 대상이 연결된 경우 `tools/list`는 **페이지마다 한 대상의 도구**를 반환하며 다음 대상에 대한 `nextCursor`를 함께 제공합니다. 

In [ ]:
mcp = GatewayMCPClient(gatewayURL, _get_inbound_token, session_id=session_id)

mcp.list_tools()

In [ ]:
all_tools = mcp.list_all_tools()
print(f"{len(all_tools)} tools across both targets:")
for t in all_tools:
    print(f"  - {t['name']}")

### 7.6단계: DEFAULT와 DYNAMIC 비교 요약

| 항목 | DEFAULT | DYNAMIC |
|---|---|---|
| `tools/list`, `prompts/list`, `resources/list`, `resources/templates/list` | Gateway 캐시에서 제공 | MCP server로 실시간 전달 |
| `tools/call`, `prompts/get`, `resources/read` | MCP server에 실시간 요청 | MCP server에 실시간 요청 |
| 기능 변경 후 `SynchronizeGatewayTargets` 필요 여부 | 예 | 아니요 |
| 시맨틱 검색(`x_amz_bedrock_agentcore_search`)과 호환 여부 | 예 | 아니요 |
| 아웃바운드 3LO OAuth와 호환 여부 | 예 | 아니요 |

## 8단계: 리소스 정리

워크숍이 끝났습니다. 아래 셀의 주석을 해제하여 생성한 모든 리소스(Gateway, OAuth2 자격 증명 공급자, Runtime, 두 Cognito 사용자 풀 및 Gateway IAM 역할)를 삭제합니다.

In [ ]:
## 8.1단계: Gateway 삭제(두 대상인 DEFAULT와 DYNAMIC도 연쇄적으로 삭제)
utils.delete_gateway(gateway_client, gatewayID)

In [ ]:
## 8.2단계: OAuth2 자격 증명 공급자 삭제
identity_client.delete_oauth2_credential_provider(name=f"{GATEWAY_NAME}-identity")

In [ ]:
## 8.3단계: AgentCore Runtime의 MCP server 삭제
!cd mcpservers && agentcore remove agent --name {MCP_SERVER_NAME} -y
!cd mcpservers && agentcore deploy -y

In [ ]:
# ## 8.4단계: Cognito CloudFormation 스택 삭제(사용자 풀, 도메인, 리소스 서버, 모든 클라이언트)
# ## 다른 실습에서 이 스택을 사용하지 않을 때 Cognito 스택 삭제
# print(f"Deleting stack {COGNITO_STACK_NAME}...")
# cfn.delete_stack(StackName=COGNITO_STACK_NAME)
# cfn.get_waiter("stack_delete_complete").wait(StackName=COGNITO_STACK_NAME)
# print(f"✅ Stack {COGNITO_STACK_NAME} deleted")

In [ ]:
## 8.5단계: Gateway IAM 역할 삭제(CFN 스택에 포함되지 않음)
utils.delete_iam_role(f"agentcore-{GATEWAY_NAME}-role")